# chapter1
## BPE代码



In [ ]:
print("hello world")

In [ ]:
# MHA
import torch
from torch import nn

class MutiHeadAttention(torch.nn.Module):
    """
    多头注意力机制（Multi-Head Attention）模块。
    """
    def __init__(self, hidden_size, num_heads):
        """
        初始化多头注意力模块。
        
        参数:
        hidden_size (int): 隐藏状态的维度。
        num_heads (int): 注意力头的数量。
        """
        super(MutiHeadAttention, self).__init__()
        self.num_heads = num_heads  # 注意力头的数量
        self.head_dim = hidden_size // num_heads  # 每个注意力头的维度

        # 查询（Query）、键（Key）和值（Value）的线性变换层
        self.q_linear = nn.Linear(hidden_size, hidden_size)
        self.k_linear = nn.Linear(hidden_size, hidden_size)
        self.v_linear = nn.Linear(hidden_size, hidden_size)

        # 输出的线性变换层
        self.o_linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden_state, attention_mask=None):
        """
        多头注意力模块的前向传播过程。
        
        参数:
        hidden_state (torch.Tensor): 输入的隐藏状态，形状为(batch_size, seq_length, hidden_size)。
        attention_mask (torch.Tensor, optional): 注意力掩码，形状为(batch_size, seq_length, seq_length)，默认为None。
        
        返回:
        torch.Tensor: 多头注意力的输出，形状为(batch_size, seq_length, hidden_size)。
        """
        batch_size = hidden_state.size(0)  # 获取批次大小

        # 将隐藏状态通过线性变换层，得到查询（Query）、键（Key）和值（Value）
        query = self.q_linear(hidden_state)
        key = self.k_linear(hidden_state)
        value = self.v_linear(hidden_state)

        # 将查询、键和值分割成多个头
        query = self.split_head(query)
        key = self.split_head(key)
        value = self.split_head(value)

        # 计算注意力分数，通过点积操作得到每个头的注意力分数
        attention_score = torch.matmul(query, key.transpose(-1, -2)) / torch.sqrt(torch.tensor(self.head_dim))

        # 如果提供了注意力掩码，则将掩码应用到注意力分数上
        if attention_mask is not None:
            attention_score += attention_mask * -1e9

        # 对注意力分数进行softmax归一化，得到注意力权重
        attention_weights = torch.softmax(attention_score, dim=-1)

        # 将注意力权重与值向量相乘，得到每个头的输出
        output = torch.matmul(attention_weights, value)

        # 将多头的输出拼接起来，恢复到原来的形状
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.head_dim * self.num_heads)

        # 通过输出线性变换层，得到最终的输出
        output = self.o_linear(output)
        return output

    def split_head(self, x):
        """
        将输入张量分割成多个头。
        
        参数:
        x (torch.Tensor): 输入张量，形状为(batch_size, seq_length, hidden_size)。
        
        返回:
        torch.Tensor: 分割后的张量，形状为(batch_size, num_heads, seq_length, head_dim)。
        """
        batch_size = x.size(0)  # 获取批次大小
        # 将输入张量的hidden_size维度分割成num_heads个子空间，每个子空间的维度为head_dim
        return x.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)


# 测试样例
if __name__ == "__main__":
    # 设置参数
    hidden_size = 64  # 隐藏状态的维度
    num_heads = 8  # 注意力头的数量
    batch_size = 2  # 批次大小
    seq_length = 5  # 序列长度

    # 创建多头注意力模块
    mha = MutiHeadAttention(hidden_size, num_heads)

    # 创建输入的隐藏状态，随机生成
    hidden_state = torch.randn(batch_size, seq_length, hidden_size)

    # 创建注意力掩码，假设第一个序列的第二个位置和第二个序列的第三个位置需要被屏蔽
    attention_mask = torch.ones(batch_size, seq_length, seq_length)
    attention_mask[0, 2, :] = -1e9  # 第一个序列的第二个位置
    attention_mask[1, 3, :] = -1e9  # 第二个序列的第三个位置

    # 前向传播，计算多头注意力的输出
    output = mha(hidden_state, attention_mask)

    # 打印输出的形状和具体内容
    print("Output shape:", output.shape)
    print("Output:", output)

# ### 代码注释说明
# 1. **类定义**：
#    - `MutiHeadAttention`类继承自`torch.nn.Module`，定义了多头注意力机制的结构和前向传播过程。

# 2. **初始化方法**：
#    - `__init__`方法中初始化了多头注意力模块的参数，包括注意力头的数量`num_heads`和每个头的维度`head_dim`。
#    - 定义了查询（Query）、键（Key）和值（Value）的线性变换层，以及输出的线性变换层。

# 3. **前向传播方法**：
#    - `forward`方法实现了多头注意力的前向传播过程。
#    - 首先通过线性变换层将输入的隐藏状态转换为查询、键和值。
#    - 然后将查询、键和值分割成多个头。
#    - 计算注意力分数，并应用注意力掩码（如果有）。
#    - 对注意力分数进行softmax归一化，得到注意力权重。
#    - 将注意力权重与值向量相乘，得到每个头的输出。
#    - 将多头的输出拼接起来，恢复到原来的形状。
#    - 最后通过输出线性变换层，得到最终的输出。

# 4. **分割头的方法**：
#    - `split_head`方法将输入张量分割成多个头，每个头的维度为`head_dim`。

# 5. **测试样例**：
#    - 设置了隐藏状态的维度、注意力头的数量、批次大小和序列长度。
#    - 创建了多头注意力模块、输入的隐藏状态和注意力掩码。
#    - 调用`forward`方法进行前向传播，计算多头注意力的输出。
#    - 打印输出的形状和具体内容，验证多头注意力模块的实现是否正确。

: 

In [ ]:
import torch
from torch import nn

class MultiGroupHead(torch.nn.Module):
    """
    多组头注意力机制（Multi-Group Head Attention）模块。
    """
    def __init__(self, hidden_size, num_heads):
        """
        初始化多组头注意力模块。
        
        参数:
        hidden_size (int): 隐藏状态的维度。
        num_heads (int): 注意力头的数量。
        """
        super(MultiGroupHead, self).__init__()
        self.num_heads = num_heads  # 注意力头的数量
        self.hidden_dim = hidden_size // num_heads  # 每个注意力头的维度

        # 查询（Query）、键（Key）和值（Value）的线性变换层
        self.q_linear = nn.Linear(hidden_size, hidden_size)
        self.k_linear = nn.Linear(hidden_size, self.hidden_dim)
        self.v_linear = nn.Linear(hidden_size, self.hidden_dim)

        # 输出的线性变换层
        self.o_linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden_state, attention_mask):
        """
        多组头注意力模块的前向传播过程。
        
        参数:
        hidden_state (torch.Tensor): 输入的隐藏状态，形状为(batch_size, seq_length, hidden_size)。
        attention_mask (torch.Tensor): 注意力掩码，形状为(batch_size, seq_length, seq_length)。
        
        返回:
        torch.Tensor: 多组头注意力的输出，形状为(batch_size, seq_length, hidden_size)。
        """
        batch_size = hidden_state.size(0)  # 获取批次大小

        # 将隐藏状态通过线性变换层，得到查询（Query）、键（Key）和值（Value）
        query = self.q_linear(hidden_state)
        key = self.k_linear(hidden_state)
        value = self.v_linear(hidden_state)

        # 将查询、键和值分割成多个头
        query = self.split_heads(query)
        key = self.split_heads(key, 1)
        value = self.split_heads(value, 1)

        # 扩展键和值的维度，使其与查询的头数一致
        key = key.expand(-1, self.num_heads, -1, -1)
        value = value.expand(-1, self.num_heads, -1, -1)

        # 计算注意力分数，通过点积操作得到每个头的注意力分数
        attention_scores = torch.matmul(query, key.transpose(-1, -2)) / torch.sqrt(torch.tensor(self.hidden_dim))

        # 如果提供了注意力掩码，则将掩码应用到注意力分数上
        if attention_mask is not None:
            attention_scores += attention_mask * -1e9

        # 对注意力分数进行softmax归一化，得到注意力权重
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # 将注意力权重与值向量相乘，得到每个头的输出
        output = torch.matmul(attention_weights, value)

        # 将多头的输出拼接起来，恢复到原来的形状
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim * self.num_heads)

        # 通过输出线性变换层，得到最终的输出
        return self.o_linear(output)

    def split_heads(self, x, head_num=None):
        """
        将输入张量分割成多个头。
        
        参数:
        x (torch.Tensor): 输入张量，形状为(batch_size, seq_length, hidden_size)。
        head_num (int, optional): 分割成的头数，默认为None，使用self.num_heads。
        
        返回:
        torch.Tensor: 分割后的张量，形状为(batch_size, num_heads, seq_length, head_dim)。
        """
        batch_size = x.size(0)  # 获取批次大小
        if head_num is None:
            head_num = self.num_heads  # 如果没有指定头数，则使用默认的头数
        # 将输入张量的hidden_size维度分割成num_heads个子空间，每个子空间的维度为hidden_dim
        return x.view(batch_size, -1, head_num, self.hidden_dim).transpose(1, 2)


# 测试样例
if __name__ == "__main__":
    # 设置参数
    hidden_size = 64  # 隐藏状态的维度
    num_heads = 8  # 注意力头的数量
    batch_size = 2  # 批次大小
    seq_length = 5  # 序列长度

    # 创建多组头注意力模块
    mgh = MultiGroupHead(hidden_size, num_heads)

    # 创建输入的隐藏状态，随机生成
    hidden_state = torch.randn(batch_size, seq_length, hidden_size)

    # 创建注意力掩码，假设第一个序列的第二个位置和第二个序列的第三个位置需要被屏蔽
    attention_mask = torch.ones(batch_size, seq_length, seq_length)
    attention_mask[0, 2, :] = -1e9  # 第一个序列的第二个位置
    attention_mask[1, 3, :] = -1e9  # 第二个序列的第三个位置

    # 前向传播，计算多组头注意力的输出
    output = mgh(hidden_state, attention_mask)

    # 打印输出的形状和具体内容
    print("Output shape:", output.shape)
    print("Output:", output)

In [ ]:
# GQA
import torch
from torch import nn

class GroupQueryAttention(torch.nn.Module):
	def __init__(self,hidden_size,num_heads,groups,drouput):
		super.__init__()
		self.hidden_size = hidden_size
		self.hidden_dim = hidden_size // num_heads
		self.groups = groups

		self.q_linear = nn.Linear(hidden_size,hidden_size)
		self.k_linear = nn.Linear(hidden_size,self.hidden_dim*self.groups)
		self.v_linear = nn.Linear(hidden_size,self.hidden_dim*self.groups)

		self.o_linear = nn.Linear(hidden_size,hidden_size)
		self.drouput = drouput
	
	def forward(self,hidden_state,attention_mask = None):

		batch_size = hidden_state.size(0)
		query = self.q_linear(hidden_size)
		key = self.k_linear(hidden_size)
		value = self.v_linear(hidden_size)

		query = query.view(batch_size,-1,self.hidden_size,self.hidden_dim)
		


